# Scenario: Finding "Ghost" Records

In [9]:
import pandas as pd
import sqlite3
# creating a dataset with duplicate patient but different IDs
patient_registry = {
    "patient_id": ["P-101", "P-102", "P-103", "P-104", "P-105"],
    "full_name": ["Li Wei", "Sarah Smith", "Li Wei", "James Bond", "Sarah Smith"],
    "dob": ["1980-05-12", "1992-11-20", "1980-05-12", "1965-01-01", "1992-11-20"],
    "last_visit": ["2026-01-10", "2026-02-15", "2026-03-01", "2026-03-05", "2026-04-10"]
}
# adding dataset to DataFrame
df_patients = pd.DataFrame(patient_registry)
# connecting the qsl
connt = sqlite3.connect(":memory:")
df_patients.to_sql("registry", connt, index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("******************************** Duplicate Detection Database *******************")
print()


******************************** Duplicate Detection Database *******************



# The "Double Vision" Check

In [15]:
# query for all data to view
all_data = "SELECT * FROM registry "
print("************************************* all data to view **************************")
display(run_query(all_data))
print()
# query to find all full_name and dob combinations that appear more than once in the registry.
double_registry = """
SELECT full_name, dob FROM registry
GROUP BY full_name, dob
HAVING COUNT(*)>1
"""
print("************************************* Double Vision **************************")
display(run_query(double_registry))

************************************* all data to view **************************


,patient_id,full_name,dob,last_visit
0,P-101,Li Wei,1980-05-12,2026-01-10
1,P-102,Sarah Smith,1992-11-20,2026-02-15
2,P-103,Li Wei,1980-05-12,2026-03-01
3,P-104,James Bond,1965-01-01,2026-03-05
4,P-105,Sarah Smith,1992-11-20,2026-04-10



************************************* Double Vision **************************


,full_name,dob
0,Li Wei,1980-05-12
1,Sarah Smith,1992-11-20


# Identifying the IDs to Merge

In [16]:
# query that shows the patient_id, full_name, and dob for only those patients identified as duplicates.
ids_to_merge = """
SELECT * FROM registry 
WHERE full_name IN (
    SELECT full_name FROM registry GROUP BY full_name HAVING COUNT(*) > 1
);
"""
print("***************************************** ids_to_merge *********************")
display(run_query(ids_to_merge))

***************************************** ids_to_merge *********************


,patient_id,full_name,dob,last_visit
0,P-101,Li Wei,1980-05-12,2026-01-10
1,P-102,Sarah Smith,1992-11-20,2026-02-15
2,P-103,Li Wei,1980-05-12,2026-03-01
3,P-105,Sarah Smith,1992-11-20,2026-04-10
